# 22. Kiwi BM25 단독 + Qwen3.8-27B 양자화 답변 미리보기

이 파일은 **Kiwi BM25만으로 검색**한 문서를 Qwen3.8-27B에 전달해 답변을 출력합니다. Dense 임베딩·RRF·재순위화·자동 채점은 사용하지 않습니다.

- 기본 설정: 청크 700자/겹침 100자, 사업별 Kiwi BM25 상위 4개, 서로 다른 사업의 질문 3개
- 생성 모델: [`Qwen/Qwen3.8-27B`](https://huggingface.co/Qwen/Qwen3.8-27B)의 [Q3_K_S 양자화 파일](https://huggingface.co/bartowski/Qwen3.8-27B-GGUF) (무료 Colab T4를 고려해 llama.cpp로 실행)
- 3비트 양자화는 원본 모델보다 답변 품질이 낮을 수 있습니다. 이번 파일은 **출력 모습 확인용**입니다.
- LangSmith 코드는 맨 아래에 준비해 두었고 기본값은 **실행 안 함**입니다.

Colab에서 **무료 T4 GPU 런타임**을 선택한 뒤 위에서 아래로 실행하세요. 모델 파일 다운로드와 llama.cpp 빌드 때문에 첫 실행은 시간이 걸립니다. Google Drive의 `MyDrive/data`에 RFP PDF/HWP/HWPX 파일, `MyDrive/data_list.csv`에 문서 목록, `MyDrive/rag_eval_dataset_3x10.json`에 질문이 있어야 합니다.


In [ ]:
%pip install -q pandas pymupdf4llm langchain-hwp-hwpx-loader langchain-text-splitters langchain-community rank_bm25 kiwipiepy huggingface_hub openai langsmith


In [ ]:
from pathlib import Path
import json
import re
import unicodedata

import pandas as pd
import pymupdf4llm
import torch
from google.colab import drive
from hwp_hwpx_parser import ExtractOptions, TableStyle
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_hwp_hwpx import HwpHwpxLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from kiwipiepy import Kiwi

drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/data')
METADATA_CSV = Path('/content/drive/MyDrive/data_list.csv')
QUESTION_JSON = Path('/content/drive/MyDrive/rag_eval_dataset_3x10.json')

if not torch.cuda.is_available():
    raise RuntimeError('Colab 메뉴에서 런타임 > 런타임 유형 변경 > GPU를 선택하세요.')
print('GPU:', torch.cuda.get_device_name(0))


## 1. 문서 불러오기

HWP/HWPX 표는 마크다운으로 추출하고, PDF도 마크다운 텍스트로 읽습니다. `data_list.csv`의 파일명으로 사업명을 연결합니다.


In [ ]:
def normalize_filename(name):
    stem = Path(str(name)).stem
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', stem)).strip()


metadata = pd.read_csv(METADATA_CSV, encoding='utf-8-sig')
required_columns = {'파일명', '사업명', '발주 기관'}
missing = required_columns - set(metadata.columns)
if missing:
    raise ValueError(f'data_list.csv에 필요한 열이 없습니다: {sorted(missing)}')

metadata_by_name = {}
for _, row in metadata.iterrows():
    key = normalize_filename(row['파일명'])
    if key in metadata_by_name:
        old = metadata_by_name[key]
        if (str(old['사업명']), str(old['발주 기관'])) != (str(row['사업명']), str(row['발주 기관'])):
            raise ValueError(f'같은 파일명에 서로 다른 사업 정보가 있습니다: {key}')
    metadata_by_name[key] = row

documents = []
for path in sorted(DATA_DIR.iterdir(), key=lambda p: unicodedata.normalize('NFKC', p.name).casefold()):
    if not path.is_file() or path.suffix.lower() not in {'.pdf', '.hwp', '.hwpx'}:
        continue
    row = metadata_by_name.get(normalize_filename(path.name))
    if row is None:
        print('CSV와 연결되지 않아 제외:', path.name)
        continue
    if path.suffix.lower() == '.pdf':
        text = pymupdf4llm.to_markdown(str(path))
    else:
        options = ExtractOptions(table_style=TableStyle.MARKDOWN)
        loader = HwpHwpxLoader(str(path), mode='single', extract_options=options)
        text = '\n\n'.join(doc.page_content for doc in loader.load())
    text = re.sub(r'\n{3,}', '\n\n', re.sub(r'[ \t]+', ' ', text.replace('\r\n', '\n'))).strip()
    if not text:
        raise ValueError(f'문서 내용이 비어 있습니다: {path.name}')
    documents.append(Document(
        page_content=text,
        metadata={
            '파일명': path.name,
            '사업명': str(row['사업명']),
            '발주 기관': str(row['발주 기관']),
        },
    ))

if not documents:
    raise ValueError(f'읽을 수 있는 문서가 없습니다: {DATA_DIR}')
print('연결된 문서:', len(documents))


## 2. 청크와 Kiwi BM25 준비

질문과 문서 모두 Kiwi 형태소 분석을 적용합니다. 검색기는 처음 쓰는 사업에 대해서만 만듭니다.


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    separators=['\n\n', '\n', ' '], chunk_size=700, chunk_overlap=100
)
chunks = splitter.split_documents(documents)
for chunk_id, chunk in enumerate(chunks):
    chunk.metadata['chunk_id'] = chunk_id
print('청크:', len(chunks))


def normalize_text(value):
    return ''.join(
        char for char in unicodedata.normalize('NFKC', str(value)).casefold()
        if char.isalnum()
    )


projects = {}
for chunk in chunks:
    key = (chunk.metadata['발주 기관'], chunk.metadata['사업명'])
    projects.setdefault(key, []).append(chunk)


def identify_project(question):
    normalized_question = normalize_text(question)
    matches = [key for key in projects if normalize_text(key[1]) in normalized_question]
    if not matches:
        raise ValueError('질문에 정식 사업명이 없습니다. 질문에 사업명을 포함하세요.')
    longest = max(len(normalize_text(key[1])) for key in matches)
    matches = [key for key in matches if len(normalize_text(key[1])) == longest]
    if len(matches) != 1:
        raise ValueError(f'사업을 하나로 식별할 수 없습니다: {matches}')
    return matches[0]


kiwi = Kiwi()
PUNCTUATION_TAGS = {'SF', 'SP', 'SS', 'SE', 'SO', 'SW'}


def kiwi_morphs(text):
    return [token.form for token in kiwi.tokenize(text)
            if token.form.strip() and token.tag not in PUNCTUATION_TAGS]


bm25_by_project = {}
BM25_TOP_K = 4


def search_kiwi_bm25(question):
    project = identify_project(question)
    if project not in bm25_by_project:
        retriever = BM25Retriever.from_documents(
            projects[project], preprocess_func=kiwi_morphs
        )
        bm25_by_project[project] = retriever
    retriever = bm25_by_project[project]
    retriever.k = BM25_TOP_K
    return project, retriever.invoke(question)


## 3. Qwen3.8 양자화 모델 준비

llama.cpp를 T4용으로 빌드하고 GGUF 파일을 다운로드합니다. 모델 파일은 약 12.7GB입니다. Colab 세션 저장 공간도 확인하세요.


In [ ]:
import subprocess
from huggingface_hub import hf_hub_download

LLAMA_DIR = Path('/content/llama.cpp')
if not LLAMA_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git', str(LLAMA_DIR)], check=True)
subprocess.run([
    'cmake', '-S', str(LLAMA_DIR), '-B', str(LLAMA_DIR / 'build'),
    '-DGGML_CUDA=ON', '-DCMAKE_CUDA_ARCHITECTURES=75',
    '-DLLAMA_BUILD_TESTS=OFF', '-DCMAKE_BUILD_TYPE=Release',
], check=True)
subprocess.run([
    'cmake', '--build', str(LLAMA_DIR / 'build'), '--config', 'Release',
    '-j', '2', '--target', 'llama-server',
], check=True)

MODEL_REPO = 'bartowski/Qwen3.8-27B-GGUF'
MODEL_FILE = 'Qwen3.8-27B-Q3_K_S.gguf'
model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE)
print('양자화 모델 준비:', MODEL_FILE)


## 4. 로컬 생성 서버 시작

llama.cpp가 GPU 메모리에 맞춰 모델을 올립니다. 답변만 출력하도록 생각 모드는 끕니다. 첫 시작에는 모델을 읽느라 시간이 걸릴 수 있습니다.


In [ ]:
import time
import requests
from openai import OpenAI

server_url = 'http://127.0.0.1:8080'
server_log = open('/content/qwen38_server.log', 'w')
server = subprocess.Popen([
    str(LLAMA_DIR / 'build/bin/llama-server'),
    '-m', str(model_path),
    '--host', '127.0.0.1', '--port', '8080',
    '--gpu-layers', 'auto', '--fit', 'on',
    '-c', '4096', '--parallel', '1',
    '--reasoning', 'off',
    '--chat-template-kwargs', '{"enable_thinking": false}',
], stdout=server_log, stderr=subprocess.STDOUT)

for _ in range(180):
    if server.poll() is not None:
        raise RuntimeError('생성 서버가 중단됐습니다. /content/qwen38_server.log를 확인하세요.')
    try:
        if requests.get(server_url + '/health', timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise TimeoutError('생성 서버 시작 시간이 초과됐습니다. /content/qwen38_server.log를 확인하세요.')

client = OpenAI(base_url=server_url + '/v1', api_key='local', timeout=300)
MODEL_NAME = client.models.list().data[0].id
print('생성 서버 준비:', MODEL_NAME)


In [ ]:
def answer_question(question):
    project, found = search_kiwi_bm25(question)
    context = '\n\n'.join(
        f'[{i}] 파일: {doc.metadata["파일명"]} / 청크: {doc.metadata["chunk_id"]}\n{doc.page_content}'
        for i, doc in enumerate(found, 1)
    )
    messages = [
        {'role': 'system', 'content': (
            '너는 RFP 문서 질문 답변 도우미다. 제공된 문서 내용만 근거로 한국어로 답한다. '
            '모르는 내용은 추측하지 말고 확인할 수 없다고 말한다. '
            '중요한 사실 뒤에는 근거 번호 [1]처럼 붙인다.'
        )},
        {'role': 'user', 'content': f'<문서>\n{context}\n</문서>\n\n<질문>\n{question}\n</질문>'},
    ]
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        max_tokens=768,
        temperature=0.7,
        top_p=0.8,
        extra_body={'top_k': 20, 'chat_template_kwargs': {'enable_thinking': False}},
    )
    answer = (response.choices[0].message.content or '').strip()
    return {
        'question': question,
        'project': project[1],
        'answer': answer,
        'chunk_ids': [doc.metadata['chunk_id'] for doc in found],
        'files': [doc.metadata['파일명'] for doc in found],
    }


## 5. 서로 다른 사업에서 3문항 미리보기

첫 3개 사업에서 질문을 하나씩 고릅니다. `PREVIEW_COUNT`를 바꿔 질문 수를 조절할 수 있습니다. 실행 결과는 화면에만 출력합니다.


In [ ]:
PREVIEW_COUNT = 3
with QUESTION_JSON.open(encoding='utf-8-sig') as file:
    question_records = json.load(file)

preview_records = []
seen_projects = set()
for record in question_records:
    if record['project'] not in seen_projects:
        preview_records.append(record)
        seen_projects.add(record['project'])
for record in question_records:
    if len(preview_records) >= PREVIEW_COUNT:
        break
    if record not in preview_records:
        preview_records.append(record)
preview_questions = [record['question'] for record in preview_records[:PREVIEW_COUNT]]
if len(preview_questions) < PREVIEW_COUNT:
    raise ValueError(f'질문 {PREVIEW_COUNT}개가 필요합니다.')

preview_results = []
for number, question in enumerate(preview_questions, 1):
    result = answer_question(question)
    preview_results.append(result)
    print(f'\n========== {number}번 질문 ==========')
    print('사업:', result['project'])
    print('질문:', result['question'])
    print('검색 청크:', result['chunk_ids'])
    print('답변:\n', result['answer'])


## 6. LangSmith 기록 (지금은 실행 안 함)

나중에 기록이 필요할 때만 아래의 `RUN_LANGSMITH`를 `True`로 바꾸세요. 기본 미리보기에는 API 키가 필요하지 않습니다. 켜면 같은 3문항을 다시 생성해 LangSmith에 질문·답변·청크 번호를 기록합니다. 채점은 하지 않습니다.


In [ ]:
RUN_LANGSMITH = False

if RUN_LANGSMITH:
    import os
    from google.colab import userdata
    from langsmith import traceable

    api_key = userdata.get('LANGSMITH_API_KEY')
    if not api_key:
        raise ValueError('Colab 비밀에 LANGSMITH_API_KEY를 등록하세요.')
    os.environ['LANGSMITH_API_KEY'] = api_key
    os.environ['LANGSMITH_TRACING'] = 'true'
    os.environ['LANGSMITH_PROJECT'] = 'rfp-kiwi-bm25-qwen38-preview'

    @traceable(name='kiwi_bm25_qwen38_answer')
    def traced_answer(question):
        return answer_question(question)

    for question in preview_questions:
        traced_answer(question)
    print('LangSmith 기록 완료:', len(preview_questions), '문항')
else:
    print('LangSmith 실행 안 함')
